# Jour 3 — Distillation de connaissances

Le teacher (embeddings Evo2 + MLP) est précis mais dépend du fait qu'un modèle à 7
milliards de paramètres ait déjà traité chaque entrée. Peut-on compresser ce qu'il a
appris dans quelque chose de minuscule qui ne regarde que des comptages de k-mers ?

**Distillation** : entraîner un petit « student » non seulement sur les étiquettes dures
0/1, mais sur les prédictions *douces* du teacher (sa probabilité, adoucie par une
température) — l'incertitude du teacher porte une information (« *dark knowledge* ») que
les étiquettes dures ne portent pas.

In [ ]:
import sys
sys.path.append("src")

import torch
from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix
from models.classifier_heads import MLPHead
from models.distillation import train_student, distillation_loss
from eval import evaluate_logits, count_params, measure_latency_torch

EMB_DIR = "../../../data/supervised/embeddings"
PROCESSED_DIR = "../../../data/supervised/processed"

X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")

teacher = MLPHead(d_in=X_train_emb.shape[1])
# NOTE : si vous avez sauvegardé les poids du teacher au notebook 02, chargez-les ici plutôt que de le ré-entraîner.

## Faire correspondre les entrées du student aux fenêtres exactes notées par le teacher

Les embeddings ont été extraits pour un sous-échantillon de fenêtres (voir
`extract_evo2_embeddings.py --max_per_split`), identifiées par `ids`. Il nous faut les
séquences brutes de ces *mêmes* fenêtres pour calculer les caractéristiques k-mer du
student.

In [ ]:
splits = load_all(PROCESSED_DIR)
train_df = splits["train"].set_index("id")
val_df = splits["val"].set_index("id")

train_seqs = train_df.loc[ids_train, "sequence"].tolist()
val_seqs = val_df.loc[ids_val, "sequence"].tolist()

X_train_kmer = torch.tensor(kmer_matrix(train_seqs, k=4), dtype=torch.float32)
X_val_kmer = torch.tensor(kmer_matrix(val_seqs, k=4), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_train_emb_t = torch.tensor(X_train_emb, dtype=torch.float32)

In [ ]:
# (ré-)entraînement rapide du teacher sur cet échantillon exact, puis on le gèle
optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)
n = X_train_emb_t.shape[0]
for epoch in range(20):
    perm = torch.randperm(n)
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            teacher(X_train_emb_t[idx]), y_train_t[idx])
        loss.backward()
        optimizer.step()

teacher.eval()
with torch.no_grad():
    teacher_logits_train = teacher(X_train_emb_t)

## Entraînons le student avec la perte hybride (entropie croisée + distillation par cibles douces)

In [ ]:
student = MLPHead(d_in=X_train_kmer.shape[1], d_hidden=32)  # volontairement minuscule

student, history = train_student(
    student, teacher_logits_train, X_train_kmer, y_train_t,
    epochs=30, temperature=4.0, alpha=0.5,
)

In [ ]:
student.eval()
with torch.no_grad():
    val_logits = student(X_val_kmer).numpy()
student_metrics = evaluate_logits(val_logits, y_val)
print("distilled student:", student_metrics)
print("params:", count_params(student), "| latency (ms/sample):",
      measure_latency_torch(student, X_val_kmer[:1]))

### Point de contrôle

Le student devrait récupérer la majeure partie de la précision du teacher tout en étant
bien plus petit (aucune dépendance à Evo2 à l'inférence — seulement des comptages de
k-mers + un petit MLP) et bien plus rapide. Essayez de faire varier `temperature` et
`alpha` dans `train_student` pour observer le compromis.

Suite : `04_compression_analysis_and_wrapup.ipynb` — mettre tous les modèles sur un seul
graphique.